# Analyse de l'IRT dynamique

Évaluation en validation croisée (5 folds) du modèle `DynamicIRT` à partir des
checkpoints entraînés (`models/dynamic_irt/folds/`), puis tracés des métriques,
des trajectoires de θ et des paramètres appris.

Le code s'exécute depuis la racine du dépôt comme depuis ce dossier : la racine
est détectée automatiquement.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from sklearn.metrics import (
    roc_auc_score,
    log_loss,
    roc_curve,
)

# Detect the repository root whether the notebook is launched from the root
# or from models/dynamic_irt/.
ROOT = Path.cwd()
while not (ROOT / "datas_management").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "models" / "dynamic_irt"))
from dynamic_irt import (
    DynamicIRT, load_data, student_kfold_split, build_grouped_tensors, predict,
)

FIGURES_DIR = ROOT / "models" / "dynamic_irt" / "figures"
FIGURES_DIR.mkdir(exist_ok=True)
FOLDS_DIR = ROOT / "models" / "dynamic_irt" / "folds"

## Métriques

In [ ]:
def compute_metrics(y_true, y_pred):
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
    return {
        "AUC":     roc_auc_score(y_true, y_pred),
        "RMSE":    rmse,
        "LogLoss": log_loss(y_true, y_pred),
    }



## Courbe ROC

In [ ]:
def plot_roc(y_true, y_pred):
    fpr, tpr, _ = roc_curve(y_true, y_pred)
    plt.figure()
    plt.plot(fpr, tpr)
    plt.plot([0, 1], [0, 1], linestyle='--')
    plt.xlabel("FPR")
    plt.ylabel("TPR")
    plt.title("ROC Curve")
    plt.savefig(FIGURES_DIR / "irt_roc_curve.pdf", bbox_inches="tight")
    plt.show()



## Trajectoires de θ

Note : le script original traçait 500 paires élève-compétence, ce qui rendait
la figure illisible. On en affiche ici un sous-ensemble (`max_students=20`).

In [ ]:
def compute_theta_trajectories(model, grouped_tensors):
    """Replays the grouped forward pass and records theta before each update."""
    model.eval()
    seq_student, seq_skill, seq_item, seq_y, seq_r, seq_mask, _ = grouped_tensors
    n_pairs, max_len = seq_item.shape

    a_pos   = F.softplus(model.a)
    alpha_k = model.alpha[seq_skill]
    mu_k    = model.mu[seq_skill]

    theta_current = model.theta0[seq_student, seq_skill].clone()

    traj_dict = {
        (seq_student[i].item(), seq_skill[i].item()): []
        for i in range(n_pairs)
    }
    pair_keys = list(traj_dict.keys())

    with torch.no_grad():
        for t in range(max_len):
            mask_t = seq_mask[:, t]
            if not mask_t.any():
                break

            j   = seq_item[:, t]
            y_t = seq_y[:,   t]
            r_t = seq_r[:,   t]

            for i in range(n_pairs):
                if mask_t[i]:
                    traj_dict[pair_keys[i]].append(theta_current[i].item())

            logit = a_pos[j] * (theta_current - model.b[j]) + model.beta * r_t
            p_t   = torch.sigmoid(logit)

            delta         = (alpha_k * (y_t - p_t) + mu_k) * mask_t.float()
            theta_current = theta_current + delta

    return traj_dict


def plot_theta_trajectory(model, grouped_tensors, max_students=20):
    """Plots theta trajectories for a subset of student-skill pairs."""
    traj_dict = compute_theta_trajectories(model, grouped_tensors)
    plt.figure(figsize=(10, 6))

    plotted = 0
    for (i, k), traj in traj_dict.items():
        if plotted >= max_students:
            break
        if len(traj) > 12:
            plt.plot(traj)
            plotted += 1

    plt.axhline(0, color="gray", linestyle="--", linewidth=0.8, label="theta = 0 (baseline)")
    plt.xlim(0, 40)
    plt.xlabel("Time step (interactions for this student-skill pair)")
    plt.ylabel("theta (latent ability, unbounded)")
    plt.title("Theta trajectories (subset of students/skills)\n"
              "Rising = learning, Falling = struggling, Drift = mu_k")
    plt.legend(fontsize="small")
    plt.savefig(FIGURES_DIR / "irt_theta_trajectories.pdf", bbox_inches="tight")
    plt.show()

## Paramètres des items et des compétences

In [ ]:
def plot_item_params(model):
    a = F.softplus(model.a).detach().cpu().numpy()
    b = model.b.detach().cpu().numpy()
    plt.figure()
    plt.hist(a, bins=30)
    plt.title("Discrimination (a) (always > 0)")
    plt.show()
    plt.figure()
    plt.hist(b, bins=30)
    plt.title("Difficulty (b)")
    plt.savefig(FIGURES_DIR / "irt_item_params.pdf", bbox_inches="tight")
    plt.show()


def plot_skill_params(model):
    """Shows the learned per-skill dynamics: alpha, mu, sigma."""
    alpha = model.alpha.detach().cpu().numpy()
    mu    = model.mu.detach().cpu().numpy()
    sigma = torch.exp(model.log_sigma).detach().cpu().numpy()

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    axes[0].bar(range(len(alpha)), alpha)
    axes[0].set_title("alpha_k - ELO learning rate per skill\n(high = ability updates fast)")
    axes[0].set_xlabel("Skill index")

    axes[1].bar(range(len(mu)), mu)
    axes[1].axhline(0, color="gray", linestyle="--")
    axes[1].set_title("mu_k - mean drift per step per skill\n(positive = students tend to improve)")
    axes[1].set_xlabel("Skill index")

    axes[2].bar(range(len(sigma)), sigma)
    axes[2].set_title("sigma_k - variability of progress per skill\n(high = inconsistent learning)")
    axes[2].set_xlabel("Skill index")

    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "irt_skill_params.pdf", bbox_inches="tight")
    plt.show()

## Validation croisée (5 folds)

In [ ]:
df = load_data()
folds    = student_kfold_split(df, k=5, seed=42)
n_items  = int(df["item"].max()  + 1)
n_skills = int(df["skill"].max() + 1)

all_metrics = []

for fold_idx, (train_df, test_df) in enumerate(folds):
    print(f"\n--- Fold {fold_idx + 1} ---")
    n_students_test = int(test_df["student"].max() + 1)

    test_tensors = build_grouped_tensors(test_df)

    checkpoint = torch.load(FOLDS_DIR / f"dynamic_irt_fold{fold_idx}.pth")

    model = DynamicIRT(
        n_students_test,
        checkpoint["n_skills"],
        checkpoint["n_items"],
    )

    state = checkpoint["model_state_dict"]
    state.pop("theta0", None)
    model.load_state_dict(state, strict=False)
    model.eval()

    p = predict(model, test_tensors).numpy()

    seq_student, seq_skill, seq_item, seq_y, seq_r, seq_mask, orig_idx = test_tensors
    valid_orig = orig_idx[seq_mask]
    order      = torch.argsort(valid_orig)
    y_true     = seq_y[seq_mask][order].numpy()

    metrics = compute_metrics(y_true, p)
    all_metrics.append(metrics)

    print(f"  AUC:     {metrics['AUC']:.4f}")
    print(f"  RMSE:    {metrics['RMSE']:.4f}")
    print(f"  LogLoss: {metrics['LogLoss']:.4f}")

print("\n=== Cross-validation summary ===")
for metric in ["AUC", "RMSE", "LogLoss"]:
    vals = [m[metric] for m in all_metrics]
    print(f"  {metric:8s}: {np.mean(vals):.4f} +/- {np.std(vals):.4f}")

## Tracés (sur le dernier fold)

In [ ]:
plot_roc(y_true, p)
plot_theta_trajectory(model, test_tensors, max_students=20)
plot_item_params(model)
plot_skill_params(model)